# 01 — Diagnóstico de Qualidade dos Dados

**Projeto Varejo** · Fase 2 · Etapa de diagnóstico

| | |
|---|---|
| **Objetivo** | Levantar o inventário completo de problemas do arquivo bruto. |
| **Entrada** | `data/raw/varejo.csv` (somente leitura, nunca alterado) |
| **Saída** | `docs/relatorio-qualidade.md` |
| **Tempo** | Menos de 10 segundos de execução |
| **Conclusão** | Registrada ao final, depois da apuração |

---

> **Regra desta etapa: nenhuma correção.**
>
> Este notebook apenas **observa e conta**. Nada é alterado, nada é imputado, nada é
> removido. As correções vêm no `02-limpeza`, e cada uma delas deve apontar para um
> achado registrado aqui.
>
> O motivo: definir o critério de correção **antes** de ver o efeito dele sobre o
> resultado é o que impede a escolha inconsciente do critério que produz o gráfico
> mais agradável.

**Dado fictício.** Este conjunto foi gerado para fins de estudo. Os problemas encontrados
são reais enquanto exercício técnico, mas não descrevem uma operação de varejo existente.

## 0. Preparação

In [ ]:
from pathlib import Path
import hashlib
import re

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

# Âncora de caminho: funciona com o Jupyter aberto na raiz ou dentro de notebooks/
RAIZ = Path.cwd()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent

BRUTO = RAIZ / "data" / "raw" / "varejo.csv"
DOCS = RAIZ / "docs"
DOCS.mkdir(exist_ok=True)

print("Raiz do projeto :", RAIZ)
print("Arquivo bruto   :", BRUTO)
print("Existe          :", BRUTO.exists())

### Coletor de achados

Cada verificação registra o que encontrou nesta lista. No final, ela vira o relatório.
Assim o documento nasce da apuração, e não da memória.

In [ ]:
ACHADOS = []

def registrar(area, achado, quantidade, unidade, observacao="", tipo="defeito"):
    """Registra um achado do diagnóstico.

    Parameters
    ----------
    area : str
        Coluna ou dimensão analisada.
    achado : str
        Descrição curta do problema encontrado.
    quantidade : int
        Volume afetado. Use 0 quando nada foi encontrado — ausência de
        problema também é informação e deve constar do relatório.
    unidade : str
        'linhas', 'células', 'valores distintos', etc.
    observacao : str
        Hipótese sobre a origem ou detalhe relevante.
    tipo : str
        'defeito'    — precisa de decisão e correção no 02-limpeza.
        'informacao' — caracteriza a base, mas não exige correção.
                       Outliers estatísticos entram aqui: estar longe da média
                       não é, por si só, defeito.
    """
    ACHADOS.append({
        "Área": area,
        "Achado": achado,
        "Quantidade": quantidade,
        "Unidade": unidade,
        "Observação": observacao,
        "Tipo": tipo,
    })
    marca = {"defeito": "!", "informacao": "i"}[tipo] if quantidade else "."
    print(f"[{marca}] {area:<14} {achado:<46} {quantidade:>5} {unidade}")

## 1. Identidade do arquivo

Antes de olhar o conteúdo, registrar **qual** arquivo foi analisado. Sem isso, o
relatório não é rastreável: daqui a três meses ninguém sabe se a apuração vale para
a versão que está no repositório.

In [ ]:
tamanho = BRUTO.stat().st_size
sha256 = hashlib.sha256(BRUTO.read_bytes()).hexdigest()

# Quebra de linha e presença de BOM: diferenças entre Windows e Ubuntu
primeiros = BRUTO.read_bytes()[:4000]
tem_bom = primeiros.startswith(b"\xef\xbb\xbf")
crlf = primeiros.count(b"\r\n")
lf_puro = primeiros.count(b"\n") - crlf

print(f"Tamanho     : {tamanho:,} bytes".replace(",", "."))
print(f"SHA-256     : {sha256}")
print(f"BOM UTF-8   : {'sim' if tem_bom else 'não'}")
print(f"Quebra linha: {'CRLF (Windows)' if crlf > lf_puro else 'LF (Unix)'}")

### Leitura defensiva

Tudo como texto (`dtype=str`). Deixar o pandas adivinhar os tipos em um arquivo sujo faz
coluna numérica virar texto em silêncio — e aí a soma retorna concatenação.

O parâmetro `keep_default_na=False` é deliberado: queremos enxergar os marcadores de
ausência **como eles estão escritos**, não convertidos automaticamente.

In [ ]:
df = pd.read_csv(
    BRUTO,
    dtype=str,
    encoding="utf-8",
    keep_default_na=False,
    na_values=[],
)

linhas, colunas = df.shape
print(f"Linhas  : {linhas}")
print(f"Colunas : {colunas}")
print()
print("Colunas encontradas:")
for i, nome in enumerate(df.columns, 1):
    print(f"  {i:>2}. {nome}")

registrar("arquivo", "linhas de dados", linhas, "linhas",
          f"SHA-256 {sha256[:12]}…", tipo="informacao")
df.head()

## 2. Inventário de ausências

Duas formas distintas de ausência convivem neste arquivo:

- **Célula vazia** — nada entre as vírgulas do CSV.
- **Marcador textual** — alguém escreveu `--`, `?`, `-`, `sem informacao` ou `N/A`.

A segunda é a perigosa: o pandas a trata como texto válido, a coluna não vira numérica,
e a falha só aparece depois, no cálculo.

In [ ]:
MARCADORES = ["--", "?", "-", "sem informacao", "N/A", "NA", "n/a", "null", "NULL"]

resumo = []
for col in df.columns:
    s = df[col].str.strip()
    vazios = int((s == "").sum())
    marcados = int(s.isin(MARCADORES).sum())
    encontrados = sorted(set(s[s.isin(MARCADORES)]))
    resumo.append({
        "coluna": col,
        "vazios": vazios,
        "marcadores": marcados,
        "ausente_total": vazios + marcados,
        "pct": round((vazios + marcados) / len(df) * 100, 1),
        "formas": ", ".join(encontrados) if encontrados else "—",
    })

ausencias = pd.DataFrame(resumo).sort_values("ausente_total", ascending=False)
display(ausencias)

for r in resumo:
    if r["ausente_total"]:
        registrar(r["coluna"], "valores ausentes (vazios + marcadores)",
                  r["ausente_total"], "células",
                  f'{r["vazios"]} vazios, {r["marcadores"]} marcadores: {r["formas"]}')

total_marcadores = int(ausencias["marcadores"].sum())
total_vazios = int(ausencias["vazios"].sum())
print()
print(f"TOTAL: {total_vazios} células vazias + {total_marcadores} com marcador textual")
print(f"       {total_vazios + total_marcadores} ausências ao todo")
print()
print("Atenção: sem keep_default_na=False, o pandas converteria 'N/A', 'NULL', 'null'")
print("e 'n/a' em nulo automaticamente, e esses casos apareceriam como célula vazia.")
print("A leitura estrita usada aqui mostra a origem real de cada ausência.")

## 3. Formatos de data

Um único formato é o esperado. Mais de um significa que a conversão precisa ser explícita
— e que uma leitura ingênua vai mandar vendas para o mês errado sem acusar erro.

In [ ]:
def classificar_data(valor):
    v = str(valor).strip()
    if v == "" or v in MARCADORES:
        return "ausente"
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", v):
        return "AAAA-MM-DD"
    if re.fullmatch(r"\d{2}/\d{2}/\d{4}", v):
        return "DD/MM/AAAA"
    if re.fullmatch(r"\d{2}-\d{2}-\d{4}", v):
        return "DD-MM-AAAA"
    return f"desconhecido ({v})"

formatos = df["data"].map(classificar_data).value_counts()
display(formatos.to_frame("ocorrências"))

principal = formatos.index[0]
divergentes = int(formatos[formatos.index != principal].sum())

if divergentes:
    outros = ", ".join(f"{k}: {v}" for k, v in formatos.items() if k != principal)
    registrar("data", "datas em formato divergente do majoritário",
              divergentes, "linhas",
              f"majoritário {principal}; divergentes — {outros}")

# Amostra dos casos divergentes, para inspeção visual
mascara = df["data"].map(classificar_data) != principal
display(df.loc[mascara, ["id_venda", "data", "cliente"]].head(8))

### Datas fora do período esperado

O projeto declara cobrir 2024. Vale confirmar — sem alterar nada, apenas contando.

In [ ]:
datas = pd.to_datetime(df["data"], format="mixed", dayfirst=True, errors="coerce")

nao_convertidas = int(datas.isna().sum())
fora_2024 = int(((datas.dt.year != 2024) & datas.notna()).sum())

print(f"Período encontrado : {datas.min().date()} a {datas.max().date()}")
print(f"Não convertidas    : {nao_convertidas}")
print(f"Fora de 2024       : {fora_2024}")

registrar("data", "datas que não convertem", nao_convertidas, "linhas")
registrar("data", "datas fora de 2024", fora_2024, "linhas",
          f"período observado: {datas.min().date()} a {datas.max().date()}")

## 4. Consistência do texto

Espaço sobrando e caixa inconsistente não geram erro nenhum. O código roda, o gráfico
sai, e o número está errado. É a classe de defeito que só um inventário sistemático
encontra.

In [ ]:
def normalizar(serie):
    """Forma canônica usada apenas para CONTAR variações — não altera o DataFrame."""
    return serie.str.strip().str.upper()

CATEGORICAS = ["categoria", "canal", "uf", "cliente"]

for col in CATEGORICAS:
    s = df[col]
    validos = s[~s.str.strip().isin(MARCADORES + [""])]
    bruto_distintos = validos.nunique()
    limpo_distintos = normalizar(validos).nunique()
    excedente = bruto_distintos - limpo_distintos

    com_espaco = int((validos != validos.str.strip()).sum())

    print(f"--- {col} ---")
    print(f"  distintos no bruto        : {bruto_distintos}")
    print(f"  distintos após padronizar : {limpo_distintos}")
    print(f"  variações excedentes      : {excedente}")
    print(f"  células com espaço sobrando: {com_espaco}")
    print()

    if excedente:
        registrar(col, "variações de grafia do mesmo valor", excedente,
                  "valores distintos",
                  f"{bruto_distintos} formas escritas para {limpo_distintos} valores reais")
    if com_espaco:
        registrar(col, "células com espaço no início ou fim", com_espaco, "células")

Detalhe das variações, coluna por coluna:

In [ ]:
for col in ["categoria", "canal", "uf"]:
    s = df[col]
    validos = s[~s.str.strip().isin(MARCADORES + [""])]
    tabela = (
        pd.DataFrame({"bruto": validos, "canonico": normalizar(validos)})
        .groupby("canonico")["bruto"]
        .agg(formas=lambda x: sorted(set(x)), ocorrencias="count")
    )
    tabela["n_formas"] = tabela["formas"].map(len)
    print(f"=== {col.upper()} — {len(tabela)} valores reais ===")
    display(tabela[["n_formas", "ocorrencias", "formas"]].sort_values("n_formas", ascending=False))

## 5. Tipos numéricos

Quais valores impedem a conversão de cada coluna numérica. O objetivo aqui é
**listar o que atrapalha**, não converter.

In [ ]:
NUMERICAS = ["quantidade", "preco_unit", "desconto_pct", "valor_total"]

for col in NUMERICAS:
    s = df[col].str.strip()
    preenchidos = s[(s != "") & (~s.isin(MARCADORES))]
    convertidos = pd.to_numeric(preenchidos.str.replace(",", ".", regex=False), errors="coerce")
    problemas = preenchidos[convertidos.isna()]

    print(f"--- {col} ---")
    print(f"  preenchidos            : {len(preenchidos)}")
    print(f"  convertem para número  : {int(convertidos.notna().sum())}")
    print(f"  NÃO convertem          : {len(problemas)}")
    if len(problemas):
        print(f"  valores problemáticos  : {dict(problemas.value_counts())}")
        registrar(col, "valores preenchidos que não convertem para número",
                  len(problemas), "células", str(dict(problemas.value_counts())))
    print()

### Faixas dos valores numéricos

In [ ]:
num = df.copy()
for col in NUMERICAS:
    s = num[col].str.strip().replace(MARCADORES + [""], np.nan)
    num[col] = pd.to_numeric(s.str.replace(",", ".", regex=False), errors="coerce")

display(num[NUMERICAS].describe().round(2))

for col in NUMERICAS:
    negativos = int((num[col] < 0).sum())
    zeros = int((num[col] == 0).sum())
    if negativos:
        registrar(col, "valores negativos", negativos, "linhas", "implausível para esta coluna")
    print(f"{col:<14} negativos: {negativos:>3}   zeros: {zeros:>3}")

fora_faixa = int((num["desconto_pct"] > 100).sum())
registrar("desconto_pct", "descontos acima de 100%", fora_faixa, "linhas")

## 6. Duplicidade

Três perguntas distintas: o identificador se repete? a linha inteira se repete?
a linha se repete ignorando o identificador?

In [ ]:
dup_id = int(df["id_venda"].duplicated().sum())
dup_total = int(df.duplicated().sum())
dup_sem_id = int(df.drop(columns=["id_venda"]).duplicated().sum())

print(f"id_venda repetido          : {dup_id}")
print(f"linhas idênticas           : {dup_total}")
print(f"linhas iguais sem o id     : {dup_sem_id}")

registrar("id_venda", "identificadores repetidos", dup_id, "linhas")
registrar("linha", "linhas integralmente duplicadas", dup_total, "linhas")
registrar("linha", "linhas duplicadas ignorando o id", dup_sem_id, "linhas")

## 7. Identidade de valor

A regra de negócio do arquivo é:

$$\text{valor\_total} = \text{quantidade} \times \text{preco\_unit} \times \left(1 - \frac{\text{desconto\_pct}}{100}\right)$$

Verificar em quantas linhas ela se sustenta é o teste mais informativo do diagnóstico.
Uma taxa alta de acerto significa que a regra é confiável — e, sendo confiável, ela
pode ser usada depois para **reconstruir** os preços ausentes, em vez de imputar pela média.

Quem não obedece à regra, com preço e desconto presentes, é candidato a erro de registro.

In [ ]:
completas = num.dropna(subset=["quantidade", "preco_unit", "desconto_pct", "valor_total"])
esperado = completas["quantidade"] * completas["preco_unit"] * (1 - completas["desconto_pct"] / 100)
diferenca = (completas["valor_total"] - esperado).abs()

TOLERANCIA = 0.05
coerentes = int((diferenca < TOLERANCIA).sum())
incoerentes = int((diferenca >= TOLERANCIA).sum())
taxa = coerentes / len(completas) * 100

print(f"Linhas com os quatro campos : {len(completas)}")
print(f"Coerentes (diferença < R$ {TOLERANCIA:.2f}) : {coerentes}  ({taxa:.1f}%)")
print(f"Incoerentes                 : {incoerentes}")

registrar("valor_total", "linhas que violam a identidade de valor",
          incoerentes, "linhas",
          f"{coerentes} de {len(completas)} coerentes ({taxa:.1f}%) — regra confiável")

suspeitas = (
    completas.assign(esperado=esperado.round(2), diferenca=diferenca.round(2))
    .nlargest(10, "diferenca")
    [["id_venda", "quantidade", "preco_unit", "desconto_pct", "valor_total", "esperado", "diferenca"]]
)
display(suspeitas)

### Registros extremos

Listados para inspeção. **Nenhuma decisão de remoção é tomada aqui** — o critério
será definido por escrito no relatório, e aplicado apenas no `02-limpeza`.

In [ ]:
extremos = num.nlargest(10, "valor_total")[
    ["id_venda", "data", "categoria", "quantidade", "preco_unit", "valor_total"]
]
display(extremos)

q1, q3 = num["valor_total"].quantile([0.25, 0.75])
iqr = q3 - q1
limite = q3 + 3 * iqr
acima = int((num["valor_total"] > limite).sum())

print(f"Q1: R$ {q1:,.2f}   Q3: R$ {q3:,.2f}   IQR: R$ {iqr:,.2f}")
print(f"Limite superior (Q3 + 3×IQR): R$ {limite:,.2f}")
print(f"Linhas acima do limite      : {acima}")

registrar("valor_total", "registros acima de Q3 + 3×IQR", acima, "linhas",
          f"limite: R$ {limite:,.2f} — dispersão alta é esperada em varejo, "
          f"NÃO é critério de remoção", tipo="informacao")

## 8. Panorama do conjunto

Não é análise exploratória — isso vem no `03`. É só a confirmação de que o arquivo
tem a forma que o README descreve.

In [ ]:
print(f"Receita total declarada : R$ {num['valor_total'].sum():,.2f}")
print(f"Período                 : {datas.min().date()} a {datas.max().date()}")
print(f"Clientes (bruto)        : {df['cliente'].nunique()}")
print(f"Clientes (padronizado)  : {normalizar(df['cliente']).nunique()}")
print(f"Categorias reais        : {normalizar(df['categoria']).nunique() - 1}")
print(f"Canais reais            : {normalizar(df['canal']).nunique() - 1}")
print(f"Unidades federativas    : {normalizar(df['uf']).nunique() - 1}")

## 9. Geração do relatório

A célula abaixo transforma a lista de achados em `docs/relatorio-qualidade.md`.

A seção **Decisões propostas** é escrita por você, no próprio arquivo gerado, e é ela
que autoriza cada correção do `02-limpeza`. Deixar o critério registrado antes de ver o
efeito é a regra central desta etapa.

In [ ]:
from datetime import datetime

achados = pd.DataFrame(ACHADOS)
encontrados = achados[achados["Quantidade"] > 0]
defeitos = encontrados[encontrados["Tipo"] == "defeito"]
informacoes = encontrados[encontrados["Tipo"] == "informacao"]

linhas_md = []
a = linhas_md.append

a("# Relatório de Qualidade dos Dados")
a("")
a("**Projeto Varejo** · Diagnóstico da base bruta")
a("")
a("> Documento gerado automaticamente por `notebooks/01-diagnostico.ipynb`.")
a("> Nenhuma correção foi aplicada nesta etapa.")
a("")
a("## Identificação do arquivo")
a("")
a("| Campo | Valor |")
a("|---|---|")
a(f"| Arquivo | `data/raw/varejo.csv` |")
a(f"| Tamanho | {tamanho:,} bytes |".replace(",", "."))
a(f"| SHA-256 | `{sha256}` |")
a(f"| Quebra de linha | {'CRLF (Windows)' if crlf > lf_puro else 'LF (Unix)'} |")
a(f"| BOM UTF-8 | {'sim' if tem_bom else 'não'} |")
a(f"| Dimensões | {linhas} linhas × {colunas} colunas |")
a(f"| Diagnóstico em | {datetime.now().strftime('%d/%m/%Y às %H:%M')} |")
a("")
a("**Natureza do dado.** Conjunto fictício, gerado para fins de estudo. "
  "Os problemas listados são reais enquanto exercício técnico, mas não descrevem "
  "uma operação de varejo existente.")
a("")
a("## Achados")
a("")
a(f"Foram executadas {len(achados)} verificações: "
  f"{len(defeitos)} apontaram defeito que exige decisão, "
  f"{len(informacoes)} caracterizam a base sem exigir correção.")
a("")
a("### Defeitos — exigem decisão e correção")
a("")
a("| # | Área | Achado | Quantidade | Unidade | Observação |")
a("|---:|---|---|---:|---|---|")
for i, (_, r) in enumerate(defeitos.iterrows(), 1):
    a(f'| {i} | `{r["Área"]}` | {r["Achado"]} | {r["Quantidade"]} | {r["Unidade"]} | {r["Observação"]} |')
a("")
a("### Informações — caracterizam a base, sem correção prevista")
a("")
a("| Área | Achado | Quantidade | Unidade | Observação |")
a("|---|---|---:|---|---|")
for _, r in informacoes.iterrows():
    a(f'| `{r["Área"]}` | {r["Achado"]} | {r["Quantidade"]} | {r["Unidade"]} | {r["Observação"]} |')
a("")
a("### Verificações sem problema encontrado")
a("")
sem = achados[achados["Quantidade"] == 0]
if len(sem):
    for _, r in sem.iterrows():
        a(f'- `{r["Área"]}` — {r["Achado"]}: nenhum caso.')
else:
    a("Nenhuma. Todas as verificações encontraram ao menos um caso.")
a("")
a("## Decisões propostas")
a("")
a("> **Preencher antes de escrever `02-limpeza.ipynb`.**")
a("> Cada correção implementada deve corresponder a uma linha desta tabela. "
  "Correção sem decisão registrada não entra no pipeline.")
a("")
a("| # | Achado | Decisão | Justificativa |")
a("|---|---|---|---|")
for i, (_, r) in enumerate(defeitos.iterrows(), 1):
    a(f'| {i} | `{r["Área"]}` — {r["Achado"]} | *a definir* | *a definir* |')
a("")
a("## Verificação pós-limpeza")
a("")
a("Tabela a ser preenchida após executar `02-limpeza.ipynb`. "
  "É a prova de que cada correção surtiu efeito.")
a("")
a("| # | Indicador | Antes | Meta | Depois |")
a("|---:|---|---:|---:|---:|")
for i, (_, r) in enumerate(defeitos.iterrows(), 1):
    a(f'| {i} | `{r["Área"]}` — {r["Achado"]} | {r["Quantidade"]} | 0 | |')
a("")

destino = DOCS / "relatorio-qualidade.md"
destino.write_text("\n".join(linhas_md), encoding="utf-8")

print(f"Relatório gravado em: {destino}")
print(f"Verificações executadas  : {len(achados)}")
print(f"Defeitos a decidir       : {len(defeitos)}")
print(f"Informações registradas  : {len(informacoes)}")

## Conclusão do diagnóstico

*Escreva aqui, com suas palavras, o que a apuração mostrou.* Sugestão de estrutura:

1. **Quais são os três problemas mais graves** e por que são os mais graves.
2. **Qual achado foi surpresa** — algo que você não esperava encontrar.
3. **O que o diagnóstico autoriza** — por exemplo, se a identidade de valor confere em
   uma taxa alta, ela pode ser usada para reconstruir os preços ausentes.
4. **O que fica pendente de decisão** antes de começar a limpeza.

Escrever esta conclusão é parte do exercício. Notebook que termina no último gráfico é
registro de exploração; notebook que termina em conclusão escrita é documento.

---

**Próximo passo:** preencher a coluna *Decisão* em `docs/relatorio-qualidade.md`
e só então criar `notebooks/02-limpeza.ipynb`.